# IMPORTS

In [91]:
import glob
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np
from sklearn import tree
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier,export_graphviz
from sklearn.model_selection import train_test_split

# FILTERS

In [92]:
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def butter_highpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='highpass', analog=False)
    y = filtfilt(b, a, data)
    return y

def bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data)

# Feature Extraction

In [93]:
def feature_extraction(data, state):
    df = data['filtered_EA'].dropna().values    
    return {
        'mean': [np.mean(df)],
        'std': [np.std(df)],
        'max': [np.max(df)],
        'min': [np.min(df)],
        'median': [np.median(df)],
        'label': state
    }

# EA Detection

In [94]:
def ea_detection(csv_file_path):
    # Load data
    df = pd.read_csv(csv_file_path)
    df['time[s]'] = (df['LocalTimestamp'] - df['LocalTimestamp'].iloc[0])

    df = df.loc[(df['time[s]'] > 120) & (df['time[s]']  < (df['time[s]'].iloc[-1])-120)]

   
    if 'SA' in df.columns:
        ea_raw = df['SA'].astype(float)
    else:
        ea_raw = df['SR'].astype(float)

    time = df['time[s]']
    sampling_rate = 15          
    
    # ea_filtered = butter_lowpass_filter(ea_raw, 0.5, sampling_rate)

    df['filtered_EA'] = ea_raw

    #gaussian not necessary for ea
    # first cut first and last set of seconds
    #emotibit, collecting data on forehead

    # plt.figure(figsize=(14, 5))
    # plt.plot(time, ea_raw, color = 'red')
    # plt.title("Raw EA")
    # plt.xlabel("Time (s)")
    # plt.ylabel("EA")
    # #plt.xlim(120, (df['time[s]'].iloc[-1])-120)
    # plt.legend()
    # plt.grid(True, alpha=0.3)
    # plt.tight_layout()
    # plt.show()

    # plt.figure(figsize=(14, 5))
    # plt.plot(time, smoothed, color = 'red')
    # plt.title("Filtered EA")
    # plt.xlabel("Time (s)")
    # plt.ylabel("EA")
    # #plt.xlim(120, (df['time[s]'].iloc[-1])-120)
    # plt.legend()
    # plt.grid(True, alpha=0.3)
    # plt.tight_layout()
    # plt.show()

    return df


# ML Model

In [95]:
def decision_tree(frames):
    X = frames.drop('label', axis=1) #drops 'label' column
    y = frames['label']

    #splits into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=3)

    #trains model?
    dt_model.fit(X_train, y_train)

    #predicts 'y' values with test 'x' values
    y_pred = dt_model.predict(X_test)

    #checks accuracy of predicted 'y' against true 'y'
    acc = accuracy_score(y_test, y_pred)

    # confusion matrix
    dt_cm = confusion_matrix(y_test, y_pred, labels=dt_model.classes_)

    # precision, recall, f1 score
    print(classification_report(y_test, y_pred))

    print("Accuracy on set:", acc)

In [96]:
# Run detection

engaged_filenames = glob.glob("engaged_SA_SR/*.csv")

data = pd.DataFrame()

for file in engaged_filenames:
    df = ea_detection(file)
    fd = feature_extraction(df, 'engaged')
    data = pd.concat([data, pd.DataFrame(fd)], ignore_index = True)

relaxed_filenames = glob.glob("relaxed_SA_SR/*.csv")

for file in relaxed_filenames:
    df = ea_detection(file)
    fd = feature_extraction(df, 'relaxed')
    data = pd.concat([data, pd.DataFrame(fd)], ignore_index = True)

decision_tree(data)

              precision    recall  f1-score   support

     engaged       0.50      0.20      0.29         5
     relaxed       0.43      0.75      0.55         4

    accuracy                           0.44         9
   macro avg       0.46      0.47      0.42         9
weighted avg       0.47      0.44      0.40         9

Accuracy on set: 0.4444444444444444
